This notebook takes a folder which contains a set of csv files which are a list of candidate control sources for each pulsar, and removes duplicates while also removing sources that are too faint to be viable controls. After doing this, it knits together the separate dataframes for each pulsar in to one dataframe, which it then saves as a csv. In this case, that csv is 'candidate_controls_truncated'. 
This notebook should be run after 'cut_controls'.

IN: 'candidate_controls_by_pulsar' folder filled with csv files named a pulsar Jname where each row is a detection of a new control source, 'paper_dfv2.csv'

OUT: 'candidate_controls_truncated.csv'

In [1]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u, constants as c
import pandas as pd
import numpy as np

In [2]:
psrs = pd.read_csv('paper_dfv2.csv')

In [2]:
#creating an empty container dataframe to hold our truncated candidate control sources
container = pd.read_csv('cand_controls_bypsr/J1809-1943.csv') #this particular file path should lead to a csv containing a vasttools.source.Source.measurements dataframe
container.drop(container.index, inplace=True)

In [ ]:
for name in psrs['JNAME']:
    print(name, end="\n")
    data = pd.read_csv('cand_controls_by_pulsar/' + name + '.csv')
    coords = SkyCoord(
        ra=data["ra_deg_cont"] * u.degree, dec=data["dec_deg_cont"] * u.degree
    )
    
    #gets rid of duplicate sources by looking at angular separation
    checked_inds = []
    for i in np.arange(coords.size):
        #print(str(i), end = ",")
        if i in checked_inds:
            continue
        for j in np.arange(coords.size):
            if j in checked_inds:
                continue
            if ((coords[i].separation(coords[j]).arcsec < 10) & (i != j)):
                checked_inds.append(j)
    inds_to_delete = checked_inds
    data.drop(inds_to_delete, inplace=True)

    # Select bright sources (SNR >=8)
    snr = data["flux_peak"] / data["rms_image"]

    snr_mask = snr >= 8
    
    # Filter the data
    cut_data = data[(snr_mask)]
    
    cut_data.to_csv('truncated_ctrls_bypsr/' + name + '.csv')
    container = pd.concat([container, cut_data])

J1654-3710
J1709-3626


In [13]:
container = container[container['flux_peak']>7]

In [17]:
container.to_csv('candidate_controls_truncated.csv')